# 02 · Attention, causal masks and padding masks

**Transformer Models · Notebook 2 of 4**

## What you will build

Attention is how one position of a sequence gathers information from the others. In this notebook you write scaled
dot-product attention with the tiny tensor engine, the **causal mask** that stops a position from reading the tokens it
will be asked to predict, a **padding mask** that hides the `<pad>` filler, and the reshapes that split attention into
several heads and merge them again. Then you prove two properties by experiment: changing later tokens leaves earlier
outputs exactly unchanged, and padded positions receive exactly zero attention weight. Each proof comes with a control
that shows it can fail.

## Learning objectives

By the end of this notebook you will be able to:

1. compute $\mathrm{softmax}(QK^\top/\sqrt{d})\,V$ with a mask, and state the shape of every intermediate array;
2. explain why the scores are divided by $\sqrt{d}$, using a measurement;
3. build causal and padding masks as boolean "may attend" arrays, and combine them;
4. split a width-$D$ representation into $H$ heads and merge it back with `reshape` and `transpose`;
5. prove future-token invariance and zero weight on padding by intervention, with negative controls.

## Dependencies

**Tier 1: NumPy and Matplotlib.** The attention code uses NumPy and the tiny tensor engine (hidden recap cell). One
optional plot in section 6 uses Matplotlib, and nothing later depends on it. The notebook needs no internet access, runs
the same way in JupyterLite, on JupyterHub and in Colab, and takes a few seconds.

### How to use this notebook

Each building block follows the same rhythm: a short explanation, a tiny demo, then **✍️ Your turn**, where you type a function into a stub and run the check cell under it. The check prints ✅ when your function matches the reference and ❌, with the failing input, when it does not. Until your version passes, the notebook quietly uses the reference, so *Run all* always works; once it passes, every later cell runs on **your** code. Cells marked 🧪 are experiments: change the marked setting and run them again.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours
import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged. `Tensor` supports `+ - * / ** @`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, NumPy-style indexing, `softmax`, `log_softmax` and `backward()`; next to it are `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been checked against numerical gradients. Run the cell, and open it if you want to see again how an operation passes its gradient back.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

### One more checker: `practice_tensor`

`practice` compares plain values, but most functions in this course take and return `Tensor`s. `practice_tensor` (hidden below) runs your function and the reference on the same inputs and compares two things: the values they return, and the gradients that flow back to every input that requires one. A function that slips into raw NumPy (through `.data`, `np.exp` or `float()`, for example) can return the right numbers while cutting the gradient path, so training would silently stop updating some weights. The second comparison catches that.

In [ ]:
# ── practice_tensor: practice() for functions built from Tensor operations ──


def _tensors_in(value):
    """Every Tensor inside value, looking into tuples, lists and dicts."""
    if isinstance(value, Tensor):
        return [value]
    if isinstance(value, dict):
        value = list(value.values())
    if isinstance(value, (list, tuple)):
        return [tensor for item in value for tensor in _tensors_in(item)]
    return []


def _values_and_gradients(function, args):
    """Run function(*args). Return its output arrays and the gradients it sends back to each input Tensor."""
    inputs = [tensor for tensor in _tensors_in(args) if tensor.requires_grad]
    zero_grad(inputs)
    result = function(*args)
    outputs = list(result) if isinstance(result, tuple) else [result]
    for output in outputs:
        if not isinstance(output, Tensor):
            hint = ("; a NumPy array on the left of + or @ turns the result into an object array, so put the Tensor first"
                    if getattr(output, "dtype", None) == object else "; build the result from Tensor operations")
            raise TypeError(f"expected a Tensor but got {type(output).__name__}{hint}")
    probe = Tensor(0.0)
    for output in outputs:  # an uneven weighting, so that no gradient cancels out by symmetry
        probe = probe + (output * np.cos(np.arange(output.data.size)).reshape(output.shape)).sum()
    probe.backward()
    values = [output.data.copy() for output in outputs]
    gradients = [np.zeros_like(tensor.data) if tensor.grad is None else tensor.grad.copy() for tensor in inputs]
    zero_grad(inputs)
    return values, gradients


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """Like practice(), for functions built from Tensor operations.

    It checks the values your function returns AND the gradients that flow back
    through them, so code that slips into raw NumPy (which cuts the gradient
    path) does not pass. Returns yours once it matches, otherwise the reference.
    """
    try:
        for args in cases:
            got, got_gradients = _values_and_gradients(yours, args)
            want, want_gradients = _values_and_gradients(reference, args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got[0] if len(got) == 1 else tuple(got))}")
                print(f"   expected     {_short(want[0] if len(want) == 1 else tuple(want))}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
            if not _close(got_gradients, want_gradients, atol):
                print(f"❌ {name}: the values match, but the gradients flowing back through your function do not.")
                print("   Use Tensor operations only: .data, np.* functions and float() cut the gradient path.")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: values and gradients match the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. "
          f"From here on the notebook runs YOUR {name}.")
    return yours

## 📦 Recap: what you built earlier

The hidden cell defines the pieces from notebook 01 under their usual names (`encode`, `decode`, `embedding_lookup`
and `sinusoidal_positions`), so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.


def encode(text, stoi):
    """Return the list of token ids for the characters of text, in order."""
    return [stoi[character] for character in text]


def decode(ids, itos):
    """Return the string spelled by a sequence of token ids."""
    return "".join(itos[index] for index in ids)


def embedding_lookup(table, ids):
    """Rows of the embedding table for each id. table: Tensor (vocab, width); ids: ints of any shape."""
    return table[ids]


def sinusoidal_positions(length, width):
    """Sinusoidal position signals, shape (length, width); width must be even."""
    positions = np.arange(length)[:, None]
    rates = 1.0 / 10000 ** (np.arange(0, width, 2) / width)
    signals = np.zeros((length, width))
    signals[:, 0::2] = np.sin(positions * rates)
    signals[:, 1::2] = np.cos(positions * rates)
    return signals

In [ ]:
sentences = ["the cat likes the dog.", "the cat likes the hen.", "the birds call the frogs.", "the dog helps the cat."]
itos = ["<pad>"] + sorted(set("".join(sentences)))
stoi = {token: index for index, token in enumerate(itos)}
print(len(itos), "tokens, id 0 is", repr(itos[0]))

## 1 · Scaled dot-product attention

Each position emits three vectors: a **query** $q_i$ (what it is looking for), a **key** $k_j$ (what it offers) and a
**value** $v_j$ (what it passes on). In self-attention all three are projections of the same input, $Q = XW_Q$,
$K = XW_K$ and $V = XW_V$. Query $i$ scores every key, softmax turns each row of scores into weights, and the output is
the weighted sum of the values:

$$s_{ij} = \frac{q_i \cdot k_j}{\sqrt{d}}, \qquad a_{ij} = \frac{e^{s_{ij}}}{\sum_{j'} e^{s_{ij'}}}, \qquad \mathrm{out}_i = \sum_j a_{ij}\, v_j .$$

**Shapes.** $Q$ is $(\dots, L_q, d)$, $K$ is $(\dots, L_k, d)$ and $V$ is $(\dots, L_k, d_v)$. The scores and the weights
are $(\dots, L_q, L_k)$, one row per query, and the output is $(\dots, L_q, d_v)$. The leading axes `...` (batch, and
later heads) simply come along.

**Masks.** A boolean `mask`, broadcastable to the scores' shape, says which keys each query **may attend to**
(`True` = allowed). Blocked scores get $-10^9$ added *before* the softmax, so their weight underflows to exactly $0$ and
the remaining weights still sum to $1$.

The engine's `transpose` takes the full order of axes, so the helper `swap_last_two` below turns `(..., L, d)` into
`(..., d, L)` for any number of leading axes. Use it for $K^\top$.

In [ ]:
def swap_last_two(t):
    """(..., L, d) -> (..., d, L). The engine's transpose takes the full order of axes."""
    axes = list(range(t.ndim))
    axes[-2], axes[-1] = axes[-1], axes[-2]
    return t.transpose(*axes)


scores = Tensor([[2.0, 1.0, 0.5]])
allowed = np.array([[True, True, False]])
print("weights without a mask:", scores.softmax(axis=-1).data.round(3))
print("weights with key 2 blocked:", (scores + np.where(allowed, 0.0, -1e9)).softmax(axis=-1).data.round(3))

### ✍️ Your turn: `attention`

Write `attention(q, k, v, mask=None)` for `Tensor`s `q` of shape `(..., Lq, d)`, `k` of shape `(..., Lk, d)` and `v`
of shape `(..., Lk, dv)`. Return a tuple `(output, weights)`:

1. `scores = q @ swap_last_two(k) / np.sqrt(d)`, where `d = q.shape[-1]`;
2. if `mask` is not `None`, add `np.where(mask, 0.0, -1e9)` to the scores;
3. `weights = scores.softmax(axis=-1)`;
4. `output = weights @ v`.

*Hints:* use only `Tensor` operations, so that gradients flow. Keep the `Tensor` on the **left** when you combine it
with a NumPy array: `scores + np.where(...)`, not `np.where(...) + scores`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def attention_reference(q, k, v, mask=None):
    """Scaled dot-product attention. Returns (output, weights)."""
    scores = q @ swap_last_two(k) / np.sqrt(q.shape[-1])
    if mask is not None:
        scores = scores + np.where(mask, 0.0, -1e9)
    weights = scores.softmax(axis=-1)
    return weights @ v, weights

In [ ]:
def attention(q, k, v, mask=None):
    """Scaled dot-product attention. Returns (output, weights)."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
rng_check = np.random.default_rng(1)
q3, k3, v3 = (Tensor(rng_check.normal(size=(2, 3, 4)), requires_grad=True) for _ in range(3))
q_short = Tensor(rng_check.normal(size=(1, 2, 4)), requires_grad=True)
k_long = Tensor(rng_check.normal(size=(1, 5, 4)), requires_grad=True)
v_long = Tensor(rng_check.normal(size=(1, 5, 3)), requires_grad=True)
q4, k4, v4 = (Tensor(rng_check.normal(size=(1, 2, 3, 4)), requires_grad=True) for _ in range(3))
lower = np.tril(np.ones((3, 3), dtype=bool))
attention = practice_tensor("attention", attention, attention_reference, cases=[
    (q3, k3, v3),
    (q3, k3, v3, lower),
    (q_short, k_long, v_long, np.array([[True, True, False, True, False], [False, True, True, True, True]])),
    (q4, k4, v4, lower),  # a (3, 3) mask broadcasts over a batch axis and a head axis
])

**Expected behaviour.** Every row of `weights` sums to 1, blocked entries are exactly 0, and the output has one row
per query. The third case has 2 queries and 5 keys, so its scores are $2 \times 5$: if your code transposed the wrong
axes, it fails here. The next cell prints one batch element's weights under the lower-triangular mask.

In [ ]:
out, weights = attention(q3, k3, v3, lower)
print("weights for batch element 0 (rows are queries, columns are keys):")
print(weights.data[0].round(3))
print("row sums:", weights.data[0].sum(axis=-1))

## 2 · The causal mask

A next-token model at position $t$ is trained to predict the token at position $t + 1$. In training, the whole sentence
is fed in at once, so without a mask position $t$ could simply attend to position $t + 1$ and copy the answer. During
generation that answer does not exist yet, so a model that learned to copy fails. The **causal mask** allows key $j$ for
query $i$ only when $j \le i$:

$$\mathrm{mask}_{ij} = [\, j \le i \,], \qquad \text{a lower-triangular } (L, L) \text{ boolean matrix.}$$

In [ ]:
length = 4
for query in range(length):
    print(f"position {query} may read positions", [key for key in range(length) if key <= query])

### ✍️ Your turn: `causal_mask`

Write `causal_mask(length)`. It returns a boolean NumPy array of shape `(length, length)` that is `True` where the key
position is at most the query position (on and below the diagonal) and `False` above it.

*Hint:* `np.tril` keeps the lower triangle of a matrix; or compare `np.arange(length)[:, None]` with
`np.arange(length)[None, :]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def causal_mask_reference(length):
    """Boolean (length, length): True where key position <= query position."""
    return np.tril(np.ones((length, length), dtype=bool))

In [ ]:
def causal_mask(length):
    """Boolean (length, length): True where key position <= query position."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
causal_mask = practice("causal_mask", causal_mask, causal_mask_reference, cases=[(1,), (4,), (6,)])
print(causal_mask(4).astype(int))

## 3 · The padding mask

A batch holds sentences of different lengths, so the short ones are padded with id 0 (`<pad>`). The padded positions
still have vectors (the `<pad>` embedding plus a position signal), and without a mask real tokens would attend to that
filler. A **key-padding mask** marks which keys are real:

$$\mathrm{mask}_{b,\,\cdot,\,j} = [\, \mathrm{ids}_{b,j} \ne \mathrm{pad} \,], \qquad \text{shape } (B, 1, L),$$

where the middle axis of length 1 broadcasts over all $L$ queries. Combine it with the causal mask by logical *and*:
`causal_mask(L) & padding_mask(ids, 0)` has shape $(B, L, L)$.

With right padding and a causal mask, every pad key already lies in the future of every real query, so the causal mask
alone happens to hide it. Bidirectional attention (an encoder) and left padding have no such luck, which is why the
proof below uses attention without the causal mask.

In [ ]:
ids_check = np.array([[5, 3, 0, 0], [2, 2, 2, 1]])
print("real tokens:", ids_check != 0)
print("as keys, broadcast over queries:", (ids_check != 0)[:, None, :].shape)

### ✍️ Your turn: `padding_mask`

Write `padding_mask(token_ids, pad_id)`. `token_ids` is an integer array of shape `(batch, length)`. Return a boolean
array of shape `(batch, 1, length)` that is `True` where the token is real (not `pad_id`).

*Hint:* compare, then insert the middle axis with `[:, None, :]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def padding_mask_reference(token_ids, pad_id):
    """Boolean (batch, 1, length): True where the key is a real token, not padding."""
    return (np.asarray(token_ids) != pad_id)[:, None, :]

In [ ]:
def padding_mask(token_ids, pad_id):
    """Boolean (batch, 1, length): True where the key is a real token, not padding."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
padding_mask = practice("padding_mask", padding_mask, padding_mask_reference, cases=[
    (ids_check, 0),
    (np.array([[4, 4, 4]]), 0),                  # nothing to hide
    (np.array([[9, 1, 9], [1, 9, 9]]), 9),       # a different padding id
])
combined = causal_mask(4) & padding_mask(ids_check, 0)
print("combined mask:", combined.shape)
print(combined[0].astype(int))

**Expected behaviour.** For the first row of `ids_check`, query 0 sees key 0 and query 1 sees keys 0 and 1, exactly as
the causal mask alone would allow. The padded queries 2 and 3 see keys 0 and 1 but never a pad key; their outputs are
never used, because notebook 03's loss ignores them.

## 4 · Multi-head attention: split and merge

One attention over the full width $D$ produces one set of weights per query. **Multi-head attention** runs $H$
attentions side by side, each on its own slice of $D/H$ features, so that different heads can attend to different
positions. After projecting $x$ of shape $(B, L, D)$, the features are split and the head axis is moved forward:

$$(B, L, D) \xrightarrow{\text{reshape}} (B, L, H, D/H) \xrightarrow{\text{transpose}(0, 2, 1, 3)} (B, H, L, D/H).$$

Now `attention` treats $B$ and $H$ as leading axes. Merging reverses the steps, $(B, H, L, d_h) \to (B, L, H, d_h) \to
(B, L, H d_h)$, and an output projection $W_O$ mixes the heads. The split costs no extra multiplications: $H$ heads of
width $D/H$ do the same arithmetic as one head of width $D$. The transpose matters. Reshaping $(B, L, D)$ straight to
$(B, H, L, D/H)$ produces the right shape with the wrong grouping, mixing the features of different positions into one
head.

In [ ]:
x_demo = np.arange(12).reshape(1, 2, 6)  # 1 sequence, 2 positions, width 6
print(x_demo.reshape(1, 2, 3, 2)[0])      # position 0: three heads of width 2, then position 1

### ✍️ Your turn: `split_heads`

Write `split_heads(x, n_heads)` for a `Tensor` `x` of shape `(batch, length, width)`, with `width` divisible by
`n_heads`. Return a `Tensor` of shape `(batch, n_heads, length, width // n_heads)`.

*Hint:* `x.reshape(batch, length, n_heads, width // n_heads)`, then `.transpose(0, 2, 1, 3)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def split_heads_reference(x, n_heads):
    """(batch, length, width) -> (batch, heads, length, width // heads)."""
    batch, length, width = x.shape
    return x.reshape(batch, length, n_heads, width // n_heads).transpose(0, 2, 1, 3)

In [ ]:
def split_heads(x, n_heads):
    """(batch, length, width) -> (batch, heads, length, width // heads)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
x_check = Tensor(np.arange(24.0).reshape(2, 3, 4), requires_grad=True)
split_heads = practice_tensor("split_heads", split_heads, split_heads_reference, cases=[
    (x_check, 2),
    (Tensor(rng_check.normal(size=(1, 5, 6)), requires_grad=True), 3),
    (Tensor(rng_check.normal(size=(1, 1, 4)), requires_grad=True), 1),
])

### ✍️ Your turn: `merge_heads`

Write `merge_heads(x)`, the inverse: a `Tensor` of shape `(batch, heads, length, head_width)` becomes
`(batch, length, heads * head_width)`.

*Hint:* undo the steps of `split_heads` in reverse order.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def merge_heads_reference(x):
    """(batch, heads, length, head_width) -> (batch, length, heads * head_width)."""
    batch, heads, length, head_width = x.shape
    return x.transpose(0, 2, 1, 3).reshape(batch, length, heads * head_width)

In [ ]:
def merge_heads(x):
    """(batch, heads, length, head_width) -> (batch, length, heads * head_width)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
merge_heads = practice_tensor("merge_heads", merge_heads, merge_heads_reference, cases=[
    (Tensor(rng_check.normal(size=(2, 2, 3, 2)), requires_grad=True),),
    (Tensor(rng_check.normal(size=(1, 3, 5, 2)), requires_grad=True),),
    (Tensor(rng_check.normal(size=(1, 1, 1, 4)), requires_grad=True),),
])
round_trip = merge_heads(split_heads(x_check, 2))
assert np.array_equal(round_trip.data, x_check.data), "merge_heads(split_heads(x)) must give back x"
print("split then merge gives back x exactly:", round_trip.shape)

### Multi-head self-attention

`multi_head_attention` (given; it only calls your functions) projects $x$ to queries, keys and values, splits the heads,
inserts a head axis into the mask, attends, merges and applies the output projection. `init_attention` creates the four
$(D, D)$ projection matrices with a scale of $1/\sqrt{D}$.

In [ ]:
def init_attention(width, rng):
    """Query, key, value and output projections, each (width, width)."""
    return {name: parameter(width, width, rng=rng, scale=width ** -0.5) for name in ("Wq", "Wk", "Wv", "Wo")}


def multi_head_attention(x, params, n_heads, mask=None):
    """Self-attention with n_heads heads. x: (B, L, D). mask: boolean, broadcastable to (B, L, L).
    Returns (output (B, L, D), weights (B, heads, L, L))."""
    q = split_heads(x @ params["Wq"], n_heads)
    k = split_heads(x @ params["Wk"], n_heads)
    v = split_heads(x @ params["Wv"], n_heads)
    if mask is not None:
        mask = np.asarray(mask)[..., None, :, :]  # add a head axis: (L, L) -> (1, L, L); (B, L, L) -> (B, 1, L, L)
    out, weights = attention(q, k, v, mask)
    return merge_heads(out) @ params["Wo"], weights


D, H = 16, 4
params = init_attention(D, np.random.default_rng(1))
table = parameter(len(itos), D, rng=np.random.default_rng(2), scale=1.0)


def embed(texts):
    """Equal-length texts -> (ids (B, L), x (B, L, D)) with positions added."""
    ids = np.array([encode(text, stoi) for text in texts])
    return ids, embedding_lookup(table, ids) + sinusoidal_positions(ids.shape[1], D)

## 5 · 🧪 Experiments

### Why divide by $\sqrt{d}$?

If the entries of $q$ and $k$ have mean 0 and variance 1, then $q \cdot k$ is a sum of $d$ such products and has
variance $d$. As the width grows, raw scores spread out, softmax puts nearly all the weight on one key, and its
gradients shrink towards zero. Dividing by $\sqrt{d}$ keeps the variance at 1 for every width. The table measures the
average largest weight in a row of 8 keys.

In [ ]:
def softmax_rows(scores):
    exps = np.exp(scores - scores.max(axis=-1, keepdims=True))
    return exps / exps.sum(axis=-1, keepdims=True)


rng_scale = np.random.default_rng(0)
widths = (4, 16, 64, 256)  # 🧪 add 1024 and run again
print(" width   largest weight, unscaled   largest weight, scaled")
for d in widths:
    q = rng_scale.normal(size=(500, 1, d))
    k = rng_scale.normal(size=(500, 8, d))
    raw = (q @ k.transpose(0, 2, 1))[:, 0]
    print(f"{d:6d}   {softmax_rows(raw).max(axis=-1).mean():24.3f}   {softmax_rows(raw / np.sqrt(d)).max(axis=-1).mean():22.3f}")

**What to notice.** Unscaled, the largest weight creeps towards 1 as $d$ grows: attention becomes a hard choice of one
key, and learning slows down. Scaled, it stays near the same value for every width.

### Break the causal mask

Two sentences that share their first 18 characters go through causal self-attention together. The cell prints, for
each position, how much the two outputs differ. Run it as it is, then try the other masks in the comment.

In [ ]:
a, b = "the cat likes the dog.", "the cat likes the hen."
first_change = next(i for i, (p, q) in enumerate(zip(a, b)) if p != q)
ids_ab, x_ab = embed([a, b])
L = ids_ab.shape[1]
experiment_mask = causal_mask(L)  # 🧪 try None, or np.triu(np.ones((L, L), dtype=bool)), and run again
out_ab, _ = multi_head_attention(x_ab, params, H, experiment_mask)
gaps = np.abs(out_ab.data[0] - out_ab.data[1]).max(axis=-1)
print(f"the sentences first differ at position {first_change}")
print("largest output difference at each position:")
print(np.array2string(gaps, precision=2, max_line_width=100))

**What to notice.** With the causal mask, positions before the first change differ by exactly 0: they cannot see the
change. Without a mask, the change leaks into *every* position, including position 0. The upper-triangular mask (each
position reads only itself and the future) leaks in the opposite direction.

## 6 · Put it together: two proofs by intervention

**Proof 1, future-token invariance.** Under the causal mask, the outputs at positions before the first change are
identical for the two sentences above, while without the mask they differ (the negative control).

**Proof 2, zero weight on padding.** A short sentence padded to the length of a longer one goes through *bidirectional*
attention (no causal mask), so pads are not automatically in the future. With the padding mask, every weight on a pad
key is exactly 0, and the real positions' outputs equal those of the same sentence run alone without any padding.
Without the padding mask (the negative control), real tokens put weight on the pads and their outputs change.

In [ ]:
# Proof 1: changing later tokens leaves earlier outputs unchanged.
out_causal, _ = multi_head_attention(x_ab, params, H, causal_mask(L))
out_open, _ = multi_head_attention(x_ab, params, H)
leak_causal = np.abs(out_causal.data[0, :first_change] - out_causal.data[1, :first_change]).max()
leak_open = np.abs(out_open.data[0, :first_change] - out_open.data[1, :first_change]).max()

# Proof 2: padded keys get zero weight, and padding does not change real outputs.
short, long = "the dog helps the cat.", "the birds call the frogs."
n_real = len(short)
ids_pad = np.zeros((2, len(long)), dtype=int)
ids_pad[0, :n_real] = encode(short, stoi)
ids_pad[1] = encode(long, stoi)
x_pad = embedding_lookup(table, ids_pad) + sinusoidal_positions(len(long), D)
out_masked, weights_masked = multi_head_attention(x_pad, params, H, padding_mask(ids_pad, 0))
out_unmasked, weights_unmasked = multi_head_attention(x_pad, params, H)
_, x_alone = embed([short])
out_alone, _ = multi_head_attention(x_alone, params, H)

weight_on_pads = weights_masked.data[0, :, :, n_real:].max()
change_masked = np.abs(out_masked.data[0, :n_real] - out_alone.data[0]).max()
weight_on_pads_unmasked = weights_unmasked.data[0, :, :, n_real:].max()
change_unmasked = np.abs(out_unmasked.data[0, :n_real] - out_alone.data[0]).max()

print(f"Proof 1: before position {first_change}, the outputs differ by {leak_causal:.1e} with the causal mask "
      f"and by {leak_open:.3f} without it.")
print(f"Proof 2: largest weight on a pad key {weight_on_pads:.1e} with the padding mask, {weight_on_pads_unmasked:.3f} without;")
print(f"         real outputs move by {change_masked:.1e} with the mask and by {change_unmasked:.3f} without.")
assert leak_causal < 1e-12, "with a causal mask, later tokens must not change earlier outputs"
assert leak_open > 1e-3, "negative control: without a mask the change should leak backwards"
assert weight_on_pads == 0.0, "padded keys must receive exactly zero weight"
assert change_masked < 1e-12, "padding must not change the outputs at real positions"
assert weight_on_pads_unmasked > 1e-3 and change_unmasked > 1e-3, "negative control: without the mask, pads leak in"
assert np.allclose(weights_masked.data.sum(axis=-1), 1.0), "every row of weights must sum to 1"
assert out_causal.shape == x_ab.shape and weights_masked.shape == (2, H, len(long), len(long))
print("✅ Both invariants hold, and both controls show that the proofs could have failed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for axis, w, title in [(axes[0], weights_masked, "padding mask"), (axes[1], weights_unmasked, "no mask")]:
    axis.imshow(w.data[0, 0], cmap="viridis", vmin=0, vmax=w.data[0, 0].max())
    axis.axvline(n_real - 0.5, color="white", linestyle="--")
    axis.set_title(f"head 0, padded sentence: {title}")
    axis.set_xlabel("key position (pads right of the line)"); axis.set_ylabel("query position")
plt.tight_layout()
plt.show()

## Exercises

### ✍️ Your turn: `sliding_window_mask`

**Exercise 1.** Some long-context models let each position attend only to a recent window of keys. Write
`sliding_window_mask(length, window)`: a causal mask that also blocks keys more than `window - 1` positions back, so
query $i$ may attend to keys $i - \text{window} + 1, \dots, i$. With `window = 1` each position sees only itself.

*Hint:* combine two comparisons of `np.arange(length)[:, None]` (queries) and `np.arange(length)[None, :]` (keys) with
`&`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def sliding_window_mask_reference(length, window):
    """Causal mask that also blocks keys more than window - 1 positions back."""
    query = np.arange(length)[:, None]
    key = np.arange(length)[None, :]
    return (key <= query) & (key > query - window)

In [ ]:
def sliding_window_mask(length, window):
    """Causal mask that also blocks keys more than window - 1 positions back."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
sliding_window_mask = practice("sliding_window_mask", sliding_window_mask, sliding_window_mask_reference, cases=[
    (5, 2), (4, 1), (3, 10),
])
print(sliding_window_mask(6, 3).astype(int))

**Exercise 2 (left padding).** Pad the short sentence on the *left* instead and combine the causal and padding masks.
Which query rows now have every key blocked? Print their weights. With $-10^9$, every score in such a row is shifted by
the same amount, so the softmax ignores the mask and spreads weight over all keys, future ones included. Now change the
mask value to `-np.inf` in a copy of `attention`: the row becomes `nan`. What happens to a masked loss, or to the next
layer, when a padded position holds `nan`? (Hint: what is `0 * nan`?)

**Exercise 3 (question).** Attention weights are not explanations. Build two keys with identical value vectors and show
that moving weight from one to the other leaves the output unchanged. What would you need, beyond the weights, to claim
that a token *caused* a prediction?

## Common errors and debugging

- **`k.T` on a 3-D or 4-D tensor.** `.T` reverses *all* the axes, so `(B, L, d)` becomes `(d, L, B)`. Swap only the last
  two axes (`swap_last_two`).
- **Mask polarity.** Here `True` means "may attend". PyTorch's `nn.Transformer` masks use `True` for *blocked*
  positions, while the boolean `attn_mask` of `F.scaled_dot_product_attention` uses `True` for positions that take
  part. Check the convention of every API you use.
- **Masking after the softmax.** The remaining weights no longer sum to 1. Mask the scores, then normalise.
- **`-np.inf` as the mask value.** A row whose keys are all blocked becomes `nan`, which then spreads into the loss and
  every gradient. A large finite value keeps the row finite.
- **`reshape(B, H, L, d_h)` without the transpose.** The shape is right but each head mixes features from different
  positions. `practice_tensor` catches this through the values.
- **A `(B, L, L)` mask against `(B, H, L, L)` scores.** NumPy aligns trailing axes, so $B$ lines up with $H$. When
  $B = H$ or $B = 1$, nothing complains and the mask lands on the wrong rows. Insert the head axis explicitly, as
  `multi_head_attention` does.
- **`np.where(...) + scores`.** A NumPy array on the left of `+` turns the result into an object array. Write
  `scores + np.where(...)`.

## Summary

- Attention computes $\mathrm{softmax}(QK^\top/\sqrt{d})\,V$; each output row is a weighted average of value vectors,
  with weights that sum to 1.
- Dividing by $\sqrt{d}$ stops wide heads from saturating the softmax.
- Masks act on the scores before the softmax. The causal mask hides the future; the padding mask hides filler. Combine
  them with `&`.
- Heads come from a reshape plus a transpose, and merging undoes both.
- Interventions with controls test what a mask really does: earlier outputs ignore later tokens exactly, and pads
  receive exactly zero weight.

## Next

Notebook 03 wraps attention in a transformer block (layer normalisation, a position-wise MLP and residual connections)
and builds the training objective: next-token targets and a loss that counts only real tokens.